# Modelos Predictivos — Violencia Intrafamiliar Colombia 2015–2024

**Dataset:** `V1.csv` · ~236 000 registros · 36 columnas

| # | Pregunta de investigación | Tipo | Variable objetivo |
|---|---|---|---|
| **Q1** | ¿Pueden predecirse los casos mensuales de VIF a partir de patrones temporales? | Regresión | N° casos / mes |
| **Q2** | ¿La educación que se recibe disminuye el abuso intrafamiliar? | Clasificación | Días de Incapacidad (0 · 1 · 2) |
| **Q3** | ¿Los factores contextuales (agresor, escenario, hora, zona) clasifican la gravedad del hecho? | Clasificación | Días de Incapacidad (0 · 1 · 2) |

**Pipeline:**
1. AED enfocado
2. Ingeniería de características + Normalización
3. Análisis no supervisado (PCA, K-Means, DBSCAN)
4. Comparación de 6 modelos por pregunta
5. Evaluación con split aleatorio **y** split temporal
6. Ajuste del modelo ganador por pregunta


## 0. Setup y carga del dataset

In [ ]:
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

# Preprocessing
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA

# Models
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import LinearSVC, LinearSVR
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor

# Evaluation
from sklearn.model_selection import (train_test_split, cross_val_score,
                                      StratifiedKFold, KFold,
                                      RandomizedSearchCV)
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay,
                              mean_absolute_error, mean_squared_error, r2_score,
                              roc_auc_score)
# Clustering
from sklearn.cluster import KMeans, DBSCAN
from sklearn.metrics import silhouette_score

# PyTorch
try:
    import torch, torch.nn as nn
    from torch.utils.data import DataLoader, TensorDataset
    TORCH = True
    torch.manual_seed(42)
    print(f"PyTorch {torch.__version__} disponible ✓")
except ImportError:
    TORCH = False
    print("PyTorch no instalado — se omitirá el modelo MLP.")

warnings.filterwarnings('ignore')
os.makedirs('graficas', exist_ok=True)
SEED = 42
np.random.seed(SEED)
plt.rcParams.update({'figure.figsize': (13, 5), 'font.size': 11})
sns.set_style('whitegrid')
print("Librerías cargadas ✓")

: 

In [ ]:
df = pd.read_csv('V1.csv', sep=';', low_memory=False)
print(f"Dimensiones: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"\nColumnas:\n{list(df.columns)}")
df.head(3)

---
## Paso 1: Definición del problema

### Variable optimizable por pregunta

| Pregunta | Target | Problema | Métrica principal |
|---|---|---|---|
| Q1 | `casos_mes` (conteo agregado) | Regresión | R², MAE, RMSE |
| Q2 | `severidad_n` (0=Leve / 1=Moderado / 2=Grave) | Clasificación multiclase | F1 ponderado, AUC |
| Q3 | `severidad_n` (mismo target que Q2) | Clasificación multiclase | F1 ponderado, AUC |

**Q2 — Hipótesis central:** A mayor nivel educativo de la víctima, menor es la severidad del abuso registrado. La educación actúa como factor protector a través de tres mecanismos: mayor capacidad de reconocer y denunciar la violencia temprano, mayor acceso a redes de apoyo e instituciones, y mayor autonomía económica para salir de la situación.

### Estrategias de entrenamiento
- **Split aleatorio:** 80 % train / 20 % test con `random_state=42`. Maximiza datos de entrenamiento.
- **Split temporal:** entrenamos con 2015–2021, evaluamos con 2022–2024. Simula predicción real sobre datos futuros y evita *data leakage* temporal.

Comparar ambas estrategias permite ver si los modelos generalizan al futuro o solo memorizan el pasado.


---
## 2. Recolección y AED enfocado

Antes de modelar exploramos la distribución de las variables objetivo y el nivel de datos faltantes en las columnas que usaremos como features.

In [ ]:
# ── Distribución de la variable objetivo: severidad ──────────────────────────
mapa_sev = {
    'Cero': 0, 'Cero días': 0, 'Sin días de incapacidad': 0,
    '1 a 30': 1,
    '31 a 90': 2, 'Más de 90': 2,
    'Sin información': np.nan, 'Cero días y sin información': np.nan
}
df['severidad_n'] = (df['Días de Incapacidad Medicolegal']
                     .astype(str).str.strip().map(mapa_sev))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('AED — Variable objetivo: Severidad (Días de Incapacidad)', fontsize=13, fontweight='bold')

# Conteo por categoría
conteo = df['severidad_n'].value_counts().sort_index()
etiquetas = {0: 'Leve (0)', 1: 'Moderado (1)', 2: 'Grave (2)'}
colores = ['#2ECC71', '#F39C12', '#E74C3C']
bars = axes[0].bar([etiquetas[k] for k in conteo.index], conteo.values, color=colores, edgecolor='white')
for bar, val in zip(bars, conteo.values):
    pct = val / conteo.sum() * 100
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                 f'{val:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_title('Distribución de severidad (registros válidos)')
axes[0].set_ylabel('N° de registros')
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
axes[0].spines[['top','right']].set_visible(False)

# Nulos en columnas clave
cols_analizar = ['Escolaridad','Ciclo Vital','Estado Civil','Tipo de Discapacidad',
                 'Pertenencia Étnica','Sexo de la victima','Grupo de edad quinquenal',
                 'Rango de Hora del Hecho X 3 Horas','Zona del Hecho','Escenario del Hecho',
                 'Presunto Agresor Detallado','Factor Desencadenante de la Agresión',
                 'Mecanismo Causal de la Lesión no Fatal','Días de Incapacidad Medicolegal']
NULOS = {'Sin información','No Sabe / No Informa','No aplica'}
pct_nulos = {col: df[col].astype(str).isin(NULOS).mean()*100 for col in cols_analizar}
pct_s = pd.Series(pct_nulos).sort_values(ascending=True)
colores_bar = ['#E74C3C' if v > 30 else '#F39C12' if v > 10 else '#3498DB' for v in pct_s]
axes[1].barh(pct_s.index, pct_s.values, color=colores_bar)
axes[1].axvline(30, color='#E74C3C', linestyle='--', linewidth=0.9, alpha=0.7, label='>30% crítico')
axes[1].axvline(10, color='#F39C12', linestyle='--', linewidth=0.9, alpha=0.7, label='>10% moderado')
axes[1].set_title('% de valores indeterminados por columna\n(rojo>30%, naranja>10%, azul≤10%)')
axes[1].set_xlabel('% registros con valor desconocido')
axes[1].legend(fontsize=9)
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('graficas/aed_target_y_nulos.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Registros con severidad válida: {df['severidad_n'].notna().sum():,} / {len(df):,}")
print(f"\nDistribución:\n{conteo.rename(index={0:'Leve',1:'Moderado',2:'Grave'}).to_string()}")

---
## Paso 3: Ingeniería de características

### Lógica detrás de cada decisión

El objetivo es transformar las columnas originales del dataset (texto, rangos, categorías) en variables numéricas que los modelos puedan interpretar. Cada feature se construyó con una justificación de dominio, no solo por disponibilidad.

---

#### Q1 — ¿Cómo predecir el número de casos mensuales de violencia intrafamiliar en Colombia?

La variable objetivo es el **conteo agregado de casos por mes**. Para predecirlo se construyó una serie temporal mensual y se crearon features que capturan tres tipos de señal: tendencia, estacionalidad y shocks externos.

| Feature | Transformación | Razón |
|---|---|---|
| `t` (1 … 120) | Índice entero creciente | Captura la **tendencia lineal** del tiempo; un coeficiente positivo indica crecimiento sostenido |
| `mes_sin`, `mes_cos` | `sin(2π·mes/12)`, `cos(2π·mes/12)` | **Encoding cíclico:** sin este paso diciembre (12) y enero (1) estarían a distancia máxima cuando en realidad son meses consecutivos. El par seno/coseno los ubica como vecinos en el círculo del año |
| `trimestre` | `(mes − 1) // 3 + 1` → valores 1-4 | Las campañas institucionales (Día de la Mujer, Navidad) y los picos de violencia siguen ciclos trimestrales reconocidos en la literatura |
| `es_pandemia` | Dummy 1 si año ∈ {2020, 2021} | El confinamiento produjo una **ruptura estructural**: aumento de violencia intrahogar + subregistro por cierres de centros médicos. El modelo necesita saber que esos meses son atípicos |
| `es_post_pandemia` | Dummy 1 si año ≥ 2022 | Captura el **efecto rebote** post-COVID (aumento histórico en 2022-2023) y el nuevo nivel de base, que es distinto al período 2015-2019 |

---

#### Q2 — ¿La educación que se recibe disminuye el abuso intrafamiliar?

La variable objetivo es `severidad_n` (0=Leve, 1=Moderado, 2=Grave) como proxy de la intensidad del abuso. La feature principal es `edu_n` (nivel educativo), complementada con variables de control demográficas para aislar el efecto de la educación de otros factores como la edad o el sexo.

| Feature | Transformación | Rol en la pregunta |
|---|---|---|
| `edu_n` (0-6) | Ordinal: sin escolaridad=0 … doctorado=6 | **Feature principal.** Proxy del nivel educativo. La hipótesis es que valores altos de `edu_n` se asocian con `severidad_n` bajos |
| `sexo_n` | Mujer=1, Hombre=0, Desconocido=0.5 | **Control.** El género modifica el efecto de la educación: la autonomía que otorga la educación puede ser diferente para mujeres en contextos de dependencia económica |
| `ciclo_n` (0-5) | Ordinal por etapa vital | **Control.** La edad cambia tanto el nivel educativo alcanzable como la dinámica del abuso; necesario para no confundir el efecto de la educación con el de la edad |
| `es_menor` | Dummy: 1 si es menor de edad | **Control.** Los menores tienen acceso educativo limitado y dinámicas de abuso diferentes; su exclusión o control evita sesgo |
| `tiene_discapacidad` | Dummy: 1 si hay alguna discapacidad | **Control.** La discapacidad puede limitar tanto el acceso a la educación como la capacidad de respuesta ante el abuso |
| `civil_n` (0-4) | Ordinal: soltero=0 … viudo=4 | **Control.** El estado civil determina el tipo de vínculo con el agresor, que es independiente del nivel educativo |

---

#### Q3 — ¿Los factores contextuales del hecho (agresor, escenario, hora, zona) permiten clasificar la gravedad de la violencia?

Mismo target que Q2 (`severidad_n`), pero con features que describen **el contexto del evento**, no a la víctima. Comparar Q2 y Q3 permite responder: ¿protege más la educación de la víctima, o importa más el contexto en que ocurre el hecho?

| Feature | Transformación | Razón |
|---|---|---|
| `hora_n` (0-7) | Ordinal: franjas de 3 horas desde medianoche | La hora del hecho está correlacionada con consumo de alcohol y estado de alerta de la víctima. Franjas nocturnas muestran patrones de severidad distintos |
| `es_noche` | Dummy: 1 si hora ∈ {0-3h, 3-6h, 21-24h} | Indicador directo de violencia nocturna, que en la literatura está asociada a mayor consumo de sustancias → mayor escalada |
| `zona_n` | Dummy: cabecera municipal=1, resto=0 | Zona rural implica menor acceso inmediato a atención médica: las lesiones llegan registradas en un estado más grave al sistema de salud |
| `escenario_n` | LabelEncoder sobre top-8 escenarios + "Otro" | El escenario del hecho (vivienda, vía pública, establecimiento) cambia radicalmente las dinámicas: la vivienda facilita violencia prolongada; espacios públicos suelen generar intervención más rápida |
| `agresor_n` (1-5) | Ordinal por cercanía/intimidad del vínculo | La pareja íntima produce en promedio lesiones más graves que otros familiares |
| `factor_n` (1-3) | Ordinal: ideológico=1, emocional=2, sustancias=3 | El alcohol/drogas como factor desencadenante está asociado a mayor impulsividad y brutalidad |
| `mecanismo_n` (1-3) | Ordinal: abrasivo/térmico=1, contundente=2, cortante=3 | El mecanismo causal es el predictor más directo de severidad física |


In [ ]:
# ═══════════════════════════════════════════════════════
# ENCODINGS COMPARTIDOS
# ═══════════════════════════════════════════════════════

# ── Escolaridad ───────────────────────────────────────
mapa_edu = {
    'Ninguna':0,'Sin escolaridad':0,'No aplica':0,'Sin información':np.nan,
    'Preescolar':1,'Educación inicial y educación preescolar':1,
    'Básica primaria':2,'Educación básica primaria':2,
    'Básica secundaria':3,'Educación básica secundaria o secundaria baja':3,
    'Educación media o secundaria alta':3,
    'Tecnológica':4,'Educación técnica profesional y tecnológica':4,
    'Profesional':5,'Universitario':5,
    'Especialización, Maestría o equivalente':6,'Maestría':6,'Doctorado o equivalente':6
}
df['edu_n'] = df['Escolaridad'].astype(str).str.strip().map(mapa_edu)

# ── Severidad (ya creada) ─────────────────────────────
# df['severidad_n'] ya existe

# ── Sexo ─────────────────────────────────────────────
df['sexo_n'] = df['Sexo de la victima'].map({'Mujer':1,'Hombre':0}).fillna(0.5)

# ── Menor de edad ─────────────────────────────────────
df['es_menor'] = (df['Grupo Mayor Menor de Edad']
                  .astype(str).str.contains('Menor', case=False, na=False)
                  .astype(int))

# ── Ciclo vital ───────────────────────────────────────
mapa_ciclo = {
    'Primera infancia (0 a 5)':0,'(0 a 4)':0,
    'Infancia (6 a 11)':1,'(5 a 9)':1,'(10 a 14)':1,
    'Adolescencia (12 a 17)':2,'(12 a 17) Adolescencia':2,
    'Juventud (18 a 28)':3,'(18 a 19)':3,'(20 a 24)':3,'(25 a 29)':3,
    'Adultez (29 a 59)':4,'(30 a 34)':4,'(35 a 39)':4,'(40 a 44)':4,
    '(45 a 49)':4,'(50 a 54)':4,'(55 a 59)':4,
    'Persona mayor (60 y más)':5,'(60 a 64)':5,'(65 a 69)':5,'(70 a 74)':5,
    '(75 a 79)':5,'(80 y más)':5
}
df['ciclo_n'] = df['Ciclo Vital'].astype(str).str.strip().map(mapa_ciclo)
# Fallback: usar grupo de edad quinquenal si ciclo no mapea
df['edad_n'] = df['Grupo de edad quinquenal'].astype(str).apply(
    lambda x: int(x.split(' ')[0].replace('(','').replace(',','').split('a')[0].strip())
    if x.startswith('(') else np.nan
).clip(0, 80) / 10  # normalizado 0-8

# ── Discapacidad ──────────────────────────────────────
df['tiene_discapacidad'] = (~df['Tipo de Discapacidad']
    .astype(str).str.strip().isin({'Ninguna','Sin información','No aplica','No Sabe / No Informa'})
).astype(int)

# ── Etnia ─────────────────────────────────────────────
df['es_indigena'] = df['Pertenencia Étnica'].astype(str).str.contains(
    'Indígena|indigena', case=False, na=False).astype(int)

# ── Estado civil ──────────────────────────────────────
mapa_civil = {
    'Soltero (a)':0,'Sin información':np.nan,'No aplica':np.nan,
    'Unión libre (Concubinato)':1,'Casado (a)':2,
    'Separado (a)':3,'Divorciado (a)':3,'Viudo (a)':4
}
df['civil_n'] = df['Estado Civil'].astype(str).str.strip().map(mapa_civil)

print("Features víctima creadas ✓")
print(df[['edu_n','sexo_n','es_menor','ciclo_n','tiene_discapacidad','civil_n','severidad_n']].describe().round(2))

In [ ]:
# ═══════════════════════════════════════════════════════
# FEATURES CONTEXTUALES (Q3)
# ═══════════════════════════════════════════════════════

# ── Hora del hecho ────────────────────────────────────
mapa_hora = {
    '(0:00 a 2:59)':0,'(3:00 a 5:59)':1,'(6:00 a 8:59)':2,
    '(9:00 a 11:59)':3,'(12:00 a 14:59)':4,'(15:00 a 17:59)':5,
    '(18:00 a 20:59)':6,'(21:00 a 23:59)':7,'Sin información':np.nan
}
df['hora_n'] = df['Rango de Hora del Hecho X 3 Horas'].astype(str).str.strip().map(mapa_hora)
df['es_noche'] = df['hora_n'].isin([0,1,7]).astype(float)  # 9pm-6am
df['es_noche'] = df['es_noche'].where(df['hora_n'].notna(), np.nan)

# ── Zona ──────────────────────────────────────────────
df['zona_n'] = df['Zona del Hecho'].map({'Cabecera municipal':1}).fillna(0)
df['zona_n'] = df['zona_n'].where(~df['Zona del Hecho'].astype(str).isin({'Sin información','No aplica'}), np.nan)

# ── Escenario ─────────────────────────────────────────
escenarios_top = (df['Escenario del Hecho'].value_counts().head(8).index.tolist())
le_escenario = LabelEncoder()
df['escenario_str'] = df['Escenario del Hecho'].astype(str).apply(
    lambda x: x if x in escenarios_top else 'Otro')
df['escenario_n'] = le_escenario.fit_transform(df['escenario_str'])

# ── Presunto agresor ──────────────────────────────────
mapa_agresor = {
    'Pareja (Cónyuge, Compañero permanente)':5,
    'Ex pareja (Ex Cónyuge, Ex Compañero permanente)':4,
    'Padre':3,'Madre':3,'Padrastro':3,'Madrastra':3,
    'Hijo (a)':2,'Hermano (a)':2,
    'Otro familiar':1,'Sin información':np.nan,'No aplica':np.nan
}
df['agresor_n'] = df['Presunto Agresor Detallado'].astype(str).str.strip().map(mapa_agresor)
df['agresor_n'] = df['agresor_n'].fillna(
    df['Presunto Agresor Detallado'].astype(str).apply(
        lambda x: 1 if 'familiar' in x.lower() or 'pariente' in x.lower() else np.nan))

# ── Factor desencadenante ─────────────────────────────
mapa_factor = {
    'Intolerancia, machismo':1,'Celos':2,'Alcoholismo, Drogadicción':3,
    'Problemas económicos':1,'Problemas de convivencia':1,
    'Sin información':np.nan,'No aplica':np.nan,'No Sabe / No Informa':np.nan
}
df['factor_n'] = df['Factor Desencadenante de la Agresión'].astype(str).str.strip().map(mapa_factor)

# ── Mecanismo causal ──────────────────────────────────
mapa_mec = {
    'Cortante':3,'Cortocontundente':3,'Cortopunzante':3,'Punzante':3,
    'Contundente':2,'Trauma de miembros':2,
    'Abrasivo':1,'Químico':2,'Térmico':1,'Sin información':np.nan
}
df['mecanismo_n'] = df['Mecanismo Causal de la Lesión no Fatal'].astype(str).str.strip().map(mapa_mec)
df['mecanismo_n'] = df['mecanismo_n'].fillna(
    df['Mecanismo Causal de la Lesión no Fatal'].astype(str).apply(
        lambda x: 2 if 'contund' in x.lower() else (3 if 'cort' in x.lower() else np.nan)))

print("Features contextuales creadas ✓")
print(df[['hora_n','es_noche','zona_n','escenario_n','agresor_n','factor_n','mecanismo_n']].describe().round(2))

In [ ]:
# ═══════════════════════════════════════════════════════
# FEATURES TEMPORALES (Q1) — serie mensual
# ═══════════════════════════════════════════════════════
mapa_mes = {
    'Enero':1,'Febrero':2,'Marzo':3,'Abril':4,'Mayo':5,'Junio':6,
    'Julio':7,'Agosto':8,'Septiembre':9,'Octubre':10,'Noviembre':11,'Diciembre':12
}
df['mes_num'] = df['Mes del hecho'].map(mapa_mes)

ts = (df.dropna(subset=['mes_num'])
        .groupby(['Año del hecho','mes_num'])
        .size()
        .reset_index(name='casos'))
ts = ts.sort_values(['Año del hecho','mes_num']).reset_index(drop=True)
ts['t'] = np.arange(1, len(ts)+1)
ts['mes_sin'] = np.sin(2 * np.pi * ts['mes_num'] / 12)
ts['mes_cos'] = np.cos(2 * np.pi * ts['mes_num'] / 12)
ts['trimestre'] = ((ts['mes_num'] - 1) // 3 + 1)
ts['es_pandemia']      = ts['Año del hecho'].isin([2020, 2021]).astype(int)
ts['es_post_pandemia'] = (ts['Año del hecho'] >= 2022).astype(int)

print(f"Serie temporal: {len(ts)} meses ({ts['Año del hecho'].min()}–{ts['Año del hecho'].max()})")
print(ts.tail(6).to_string())

# Visualización de la serie
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(ts['t'], ts['casos'], color='#2C3E6B', linewidth=1.8, label='Casos reales')
ax.fill_between(ts[ts['es_pandemia']==1]['t'], 0, ts[ts['es_pandemia']==1]['casos'],
                alpha=0.25, color='#F39C12', label='Pandemia 2020-2021')
ax.fill_between(ts[ts['es_post_pandemia']==1]['t'], 0, ts[ts['es_post_pandemia']==1]['casos'],
                alpha=0.15, color='#E74C3C', label='Post-pandemia 2022+')
ax.set_title('Q1 — Serie temporal de casos mensuales de VIF (2015–2024)', fontsize=13, fontweight='bold')
ax.set_xlabel('Mes (t = 1 → ene-2015)')
ax.set_ylabel('N° de casos')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend()
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('graficas/q1_serie_temporal.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════
# NORMALIZACIÓN — StandardScaler
# ═══════════════════════════════════════════════════════
# Se aplica DESPUÉS del split para evitar data leakage
# Aquí sólo definimos las features por pregunta

FEATS_Q1 = ['t','mes_sin','mes_cos','trimestre','es_pandemia','es_post_pandemia']
FEATS_Q2 = ['edu_n','sexo_n','es_menor','ciclo_n','tiene_discapacidad','civil_n']
FEATS_Q3 = ['hora_n','es_noche','zona_n','escenario_n','agresor_n','factor_n','mecanismo_n']
TARGET    = 'severidad_n'

# Dataset limpio Q2 (drop NaN en features o target)
df_q2 = df.dropna(subset=FEATS_Q2 + [TARGET]).reset_index(drop=True)
df_q3 = df.dropna(subset=FEATS_Q3 + [TARGET]).reset_index(drop=True)

print(f"Registros disponibles Q2: {len(df_q2):,}")
print(f"Registros disponibles Q3: {len(df_q3):,}")
print(f"\nBalance de clases Q2:\n{df_q2[TARGET].value_counts().sort_index().rename({0:'Leve',1:'Moderado',2:'Grave'}).to_string()}")
print(f"\nBalance de clases Q3:\n{df_q3[TARGET].value_counts().sort_index().rename({0:'Leve',1:'Moderado',2:'Grave'}).to_string()}")

---
## Paso 4: Estrategia de entrenamiento

### Separación de datos y escenarios de evaluación

Antes de entrenar cualquier modelo se definen las reglas del juego: cómo se dividen los datos y bajo qué condiciones se evalúa cada modelo. Se usan **dos estrategias de split** para medir tanto rendimiento general como capacidad de generalización temporal.

| Estrategia | Descripción | Para qué sirve |
|---|---|---|
| **Split A — Aleatorio (80/20)** | `train_test_split` con `random_state=42`. 80 % entrenamiento, 20 % prueba, mezclados aleatoriamente | Maximiza datos de entrenamiento; mide el rendimiento en condiciones ideales |
| **Split B — Temporal** | Entrenamiento con años 2015–2021, prueba con 2022–2024 | Simula predicción real sobre datos futuros; detecta si el modelo memoriza o realmente generaliza |

Comparar ambas estrategias permite identificar si existe **data leakage temporal**: un modelo con Split A muy alto pero Split B muy bajo está memorizando el pasado, no aprendiendo patrones generalizables.

### Replicabilidad
Todos los modelos usan `random_state=42` y `np.random.seed(42)` para garantizar resultados reproducibles.

### Exploración no supervisada previa al modelado
Antes de entrenar modelos supervisados se aplica un análisis no supervisado sobre las features de Q2 (educación y perfil de la víctima). Esto permite:
- Verificar si existen **agrupaciones naturales** por nivel educativo antes de imponer etiquetas de severidad
- Detectar **outliers** que puedan sesgar el entrenamiento
- Validar que la variable `edu_n` tiene estructura discriminante real sobre `severidad_n`

Se aplican: **PCA** (reducción dimensional) → **K-Means** (clustering) → **DBSCAN** (detección de anomalías)


In [ ]:
# ── PCA sobre features Q2 ─────────────────────────────────────────────────────
X_pca = df_q2[FEATS_Q2].copy()
scaler_pca = StandardScaler()
X_pca_sc = scaler_pca.fit_transform(X_pca)

pca = PCA(random_state=SEED)
pca.fit(X_pca_sc)

varianza_acum = np.cumsum(pca.explained_variance_ratio_)
n_comp_95 = np.argmax(varianza_acum >= 0.95) + 1

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('PCA — Features del perfil de víctima (Q2)', fontsize=13, fontweight='bold')

# Varianza explicada
axes[0].bar(range(1, len(pca.explained_variance_ratio_)+1),
            pca.explained_variance_ratio_, color='#2C3E6B', alpha=0.8)
axes[0].plot(range(1, len(varianza_acum)+1), varianza_acum,
             color='#E74C3C', marker='o', linewidth=2, markersize=5)
axes[0].axhline(0.95, color='#F39C12', linestyle='--', linewidth=1.5, label='95% varianza')
axes[0].axvline(n_comp_95, color='#F39C12', linestyle=':', linewidth=1.5)
axes[0].set_title(f'Varianza explicada (95% con {n_comp_95} componentes)')
axes[0].set_xlabel('Componente principal')
axes[0].set_ylabel('Varianza explicada')
axes[0].legend()
axes[0].spines[['top','right']].set_visible(False)

# Dispersión PC1 vs PC2 coloreada por severidad
pca2 = PCA(n_components=2, random_state=SEED)
coords = pca2.fit_transform(X_pca_sc)
sample_idx = np.random.choice(len(coords), size=min(5000, len(coords)), replace=False)
colores_sev = {0:'#2ECC71', 1:'#F39C12', 2:'#E74C3C'}
for sev, label in {0:'Leve',1:'Moderado',2:'Grave'}.items():
    mask = df_q2[TARGET].iloc[sample_idx].values == sev
    axes[1].scatter(coords[sample_idx][mask, 0], coords[sample_idx][mask, 1],
                    c=colores_sev[sev], label=label, alpha=0.5, s=15, edgecolors='none')
axes[1].set_title('PC1 vs PC2 (muestra 5 000) — coloreado por severidad')
axes[1].set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]*100:.1f}%)')
axes[1].set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]*100:.1f}%)')
axes[1].legend(markerscale=2)
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('graficas/pca_q2.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Componentes para 95% de varianza: {n_comp_95}")

In [ ]:
# ── K-Means + Silhouette Score ────────────────────────────────────────────────
sample_km = np.random.choice(len(X_pca_sc), size=min(20000, len(X_pca_sc)), replace=False)
X_km = X_pca_sc[sample_km]

silhouette_scores = []
inertias = []
K_range = range(2, 9)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = km.fit_predict(X_km)
    silhouette_scores.append(silhouette_score(X_km, labels, sample_size=3000))
    inertias.append(km.inertia_)

mejor_k = K_range[np.argmax(silhouette_scores)]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('K-Means — Búsqueda del número óptimo de clusters', fontsize=13, fontweight='bold')

axes[0].plot(K_range, silhouette_scores, marker='o', color='#2C3E6B', linewidth=2)
axes[0].axvline(mejor_k, color='#E74C3C', linestyle='--', linewidth=1.5, label=f'Óptimo k={mejor_k}')
axes[0].set_title('Silhouette Score por k')
axes[0].set_xlabel('Número de clusters (k)')
axes[0].set_ylabel('Silhouette Score')
axes[0].legend()
axes[0].spines[['top','right']].set_visible(False)

axes[1].plot(K_range, inertias, marker='s', color='#E74C3C', linewidth=2)
axes[1].set_title('Inercia (Método del codo)')
axes[1].set_xlabel('Número de clusters (k)')
axes[1].set_ylabel('Inercia')
axes[1].spines[['top','right']].set_visible(False)

# Visualización del mejor k en PCA 2D
km_final = KMeans(n_clusters=mejor_k, random_state=SEED, n_init=10)
cluster_labels = km_final.fit_predict(X_km)
paleta = plt.cm.Set1(np.linspace(0, 0.8, mejor_k))
for c in range(mejor_k):
    mask_c = cluster_labels == c
    axes[2].scatter(coords[sample_km][mask_c, 0], coords[sample_km][mask_c, 1],
                    c=[paleta[c]], label=f'Cluster {c+1}', alpha=0.4, s=12, edgecolors='none')
axes[2].set_title(f'Clusters K-Means (k={mejor_k}) en espacio PCA')
axes[2].set_xlabel('PC1')
axes[2].set_ylabel('PC2')
axes[2].legend(markerscale=2, fontsize=9)
axes[2].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('graficas/kmeans_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Mejor número de clusters: {mejor_k} (Silhouette = {max(silhouette_scores):.4f})")
print(f"Silhouette scores por k: { {k: round(s,4) for k,s in zip(K_range, silhouette_scores)} }")

In [ ]:
# ── DBSCAN — Detección de outliers ───────────────────────────────────────────
# Aplicado sobre PCA 2D (muestra para velocidad)
sample_db = np.random.choice(len(coords), size=min(8000, len(coords)), replace=False)
X_db = coords[sample_db]

db = DBSCAN(eps=0.3, min_samples=20)
db_labels = db.fit_predict(X_db)
n_clusters_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
n_outliers = (db_labels == -1).sum()

fig, ax = plt.subplots(figsize=(10, 6))
colores_db = {-1:'#E74C3C'}
paleta_db = plt.cm.tab10(np.linspace(0, 0.9, max(1, n_clusters_db)))
for c in sorted(set(db_labels)):
    mask_d = db_labels == c
    color = '#E74C3C' if c == -1 else paleta_db[c % len(paleta_db)]
    label = f'Outliers ({n_outliers})' if c == -1 else f'Cluster {c+1}'
    ax.scatter(X_db[mask_d, 0], X_db[mask_d, 1],
               c=[color], label=label, alpha=0.5 if c != -1 else 0.8,
               s=10 if c != -1 else 25, edgecolors='none')
ax.set_title(f'DBSCAN — {n_clusters_db} clusters + {n_outliers} outliers detectados\n(eps=0.3, min_samples=20, muestra 8 000)', fontsize=12, fontweight='bold')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend(markerscale=3, fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('graficas/dbscan.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"DBSCAN: {n_clusters_db} clusters, {n_outliers} outliers ({n_outliers/len(X_db)*100:.1f}% de la muestra)")

---
## Paso 5: Seleccionar algoritmo

### Criterios de selección

Se seleccionaron 6 algoritmos para cada pregunta, balanceando **capacidad** (qué tan complejo puede ser el patrón que aprende), **velocidad** (tiempo de entrenamiento con ~80–236 k registros) y **precisión esperada** (según el tipo de problema: regresión vs. clasificación multiclase).

| Algoritmo | Capacidad | Velocidad | Por qué se incluye |
|---|---|---|---|
| **Regresión Lineal / Logística** | Baja | Muy rápida | Baseline interpretable; si los demás no lo superan, el problema es lineal |
| **Árbol de Decisión** | Media | Rápida | Captura no linealidades simples; muy interpretable via reglas |
| **Random Forest** | Alta | Media | Ensemble robusto al ruido y los outliers; maneja bien variables categóricas codificadas |
| **SVM (Lineal)** | Media | Lenta en n>50k | Buena generalización con margen máximo; útil para clases desbalanceadas |
| **K-NN** | Alta | Rápida en inferencia | Sin supuestos distribucionales; detecta patrones locales que los demás pierden |
| **PyTorch MLP** | Muy alta | Media (GPU opcional) | Red neuronal de 2 capas ocultas; captura interacciones no lineales complejas entre features |

### Estructura de ejecución

Cada pregunta se ejecuta bajo los dos splits definidos en el Paso 4. Los resultados se consolidan en tablas comparativas por pregunta.


In [ ]:
# ═══════════════════════════════════════════════════════
# CLASE MLP PYTORCH
# ═══════════════════════════════════════════════════════
class MLP(nn.Module):
    def __init__(self, n_in, n_out, task='clf'):
        super().__init__()
        self.task = task
        self.net = nn.Sequential(
            nn.Linear(n_in, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64),  nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, n_out)
        )
    def forward(self, x):
        return self.net(x)

def entrenar_mlp(X_tr, y_tr, X_te, y_te, task='clf', epochs=100, lr=1e-3):
    """Entrena un MLP PyTorch y devuelve métricas."""
    if not TORCH:
        return None
    device = 'cpu'
    n_out = len(np.unique(y_tr)) if task == 'clf' else 1
    model = MLP(X_tr.shape[1], n_out, task).to(device)
    
    Xtr_t = torch.FloatTensor(X_tr).to(device)
    Xte_t = torch.FloatTensor(X_te).to(device)
    ytr_t = torch.LongTensor(y_tr.astype(int)).to(device) if task=='clf'             else torch.FloatTensor(y_tr).unsqueeze(1).to(device)
    
    criterion = nn.CrossEntropyLoss() if task=='clf' else nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=10, factor=0.5)
    
    dataset = TensorDataset(Xtr_t, ytr_t)
    loader  = DataLoader(dataset, batch_size=512, shuffle=True)
    
    best_loss = float('inf')
    best_state = None
    patience_cnt = 0
    
    for ep in range(epochs):
        model.train()
        for xb, yb in loader:
            optimizer.zero_grad()
            out = model(xb)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
        
        # Validación
        model.eval()
        with torch.no_grad():
            out_val = model(Xte_t)
            val_loss = criterion(out_val, ytr_t[:len(Xte_t)] if False else
                                 (torch.LongTensor(y_te.astype(int)).to(device) if task=='clf'
                                  else torch.FloatTensor(y_te).unsqueeze(1).to(device))).item()
        scheduler.step(val_loss)
        if val_loss < best_loss:
            best_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_cnt = 0
        else:
            patience_cnt += 1
        if patience_cnt > 20:
            break
    
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        preds_raw = model(Xte_t)
        if task == 'clf':
            preds = preds_raw.argmax(dim=1).numpy()
        else:
            preds = preds_raw.squeeze().numpy()
    return preds

# ═══════════════════════════════════════════════════════
# FUNCIÓN: comparar modelos de REGRESIÓN
# ═══════════════════════════════════════════════════════
def comparar_regresion(X_tr, X_te, y_tr, y_te, nombre_split, label_q):
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)
    
    modelos = {
        'Regresión Lineal': LinearRegression(),
        'Árbol de Decisión': DecisionTreeRegressor(max_depth=6, random_state=SEED),
        'Random Forest':     RandomForestRegressor(n_estimators=100, max_depth=8, random_state=SEED, n_jobs=-1),
        'SVR (lineal)':      LinearSVR(max_iter=5000, random_state=SEED),
        'K-NN':              KNeighborsRegressor(n_neighbors=5, n_jobs=-1),
    }
    
    resultados = {}
    for nombre, modelo in modelos.items():
        modelo.fit(X_tr_s, y_tr)
        pred = modelo.predict(X_te_s)
        resultados[nombre] = {
            'MAE':  mean_absolute_error(y_te, pred),
            'MSE':  mean_squared_error(y_te, pred),
            'RMSE': np.sqrt(mean_squared_error(y_te, pred)),
            'R2':   r2_score(y_te, pred),
        }
    
    if TORCH:
        pred_mlp = entrenar_mlp(X_tr_s, y_tr, X_te_s, y_te, task='reg')
        if pred_mlp is not None:
            resultados['PyTorch MLP'] = {
                'MAE':  mean_absolute_error(y_te, pred_mlp),
                'MSE':  mean_squared_error(y_te, pred_mlp),
                'RMSE': np.sqrt(mean_squared_error(y_te, pred_mlp)),
                'R2':   r2_score(y_te, pred_mlp),
            }
    
    df_res = pd.DataFrame(resultados).T.round(4)
    print(f"\n{'='*55}")
    print(f"{label_q} — {nombre_split}")
    print('='*55)
    print(df_res.to_string())
    return df_res

# ═══════════════════════════════════════════════════════
# FUNCIÓN: comparar modelos de CLASIFICACIÓN
# ═══════════════════════════════════════════════════════
def comparar_clasificacion(X_tr, X_te, y_tr, y_te, nombre_split, label_q,
                            graficar_cm=True, fname_prefix='model'):
    y_tr = y_tr.astype(int)
    y_te = y_te.astype(int)
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)
    
    modelos = {
        'Reg. Logística':    LogisticRegression(max_iter=1000, C=1.0, random_state=SEED),
        'Árbol de Decisión': DecisionTreeClassifier(max_depth=8, random_state=SEED),
        'Random Forest':     RandomForestClassifier(n_estimators=100, max_depth=8,
                                                     class_weight='balanced', random_state=SEED, n_jobs=-1),
        'SVM (lineal)':      LinearSVC(max_iter=5000, class_weight='balanced', random_state=SEED),
        'K-NN':              KNeighborsClassifier(n_neighbors=7, n_jobs=-1),
    }
    
    resultados = {}
    preds_dict = {}
    for nombre, modelo in modelos.items():
        modelo.fit(X_tr_s, y_tr)
        pred = modelo.predict(X_te_s)
        preds_dict[nombre] = pred
        resultados[nombre] = {
            'Accuracy': accuracy_score(y_te, pred),
            'F1 (pond.)': f1_score(y_te, pred, average='weighted', zero_division=0),
            'F1 Leve':    f1_score(y_te, pred, labels=[0], average='micro', zero_division=0),
            'F1 Mod.':    f1_score(y_te, pred, labels=[1], average='micro', zero_division=0),
            'F1 Grave':   f1_score(y_te, pred, labels=[2], average='micro', zero_division=0),
        }
    
    if TORCH:
        pred_mlp = entrenar_mlp(X_tr_s, y_tr, X_te_s, y_te, task='clf')
        if pred_mlp is not None:
            preds_dict['PyTorch MLP'] = pred_mlp
            resultados['PyTorch MLP'] = {
                'Accuracy':  accuracy_score(y_te, pred_mlp),
                'F1 (pond.)':f1_score(y_te, pred_mlp, average='weighted', zero_division=0),
                'F1 Leve':   f1_score(y_te, pred_mlp, labels=[0], average='micro', zero_division=0),
                'F1 Mod.':   f1_score(y_te, pred_mlp, labels=[1], average='micro', zero_division=0),
                'F1 Grave':  f1_score(y_te, pred_mlp, labels=[2], average='micro', zero_division=0),
            }
    
    df_res = pd.DataFrame(resultados).T.round(4)
    print(f"\n{'='*55}")
    print(f"{label_q} — {nombre_split}")
    print('='*55)
    print(df_res.to_string())
    
    # Matriz de confusión del mejor modelo (por F1 ponderado)
    if graficar_cm:
        mejor = df_res['F1 (pond.)'].idxmax()
        cm = confusion_matrix(y_te, preds_dict[mejor])
        fig, ax = plt.subplots(figsize=(6, 5))
        disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                       display_labels=['Leve','Moderado','Grave'])
        disp.plot(ax=ax, colorbar=False, cmap='Blues')
        ax.set_title(f'{label_q} — {nombre_split}\nMejor modelo: {mejor}', fontsize=11, fontweight='bold')
        plt.tight_layout()
        plt.savefig(f'graficas/{fname_prefix}_{nombre_split.replace(" ","_")}_cm.png', dpi=150, bbox_inches='tight')
        plt.show()
    
    return df_res

print("Funciones de modelado definidas ✓")

---
### Paso 5.1 — Q1: ¿Cómo predecir el número de casos mensuales de violencia intrafamiliar?

**Target:** `casos` (conteo mensual agregado)  
**Features:** `t`, `mes_sin`, `mes_cos`, `trimestre`, `es_pandemia`, `es_post_pandemia`  
**Modelos:** Regresión Lineal · Árbol de Decisión · Random Forest · SVR · K-NN · PyTorch MLP  
**Splits:** (A) Aleatorio 80/20 · (B) Temporal 2015–2021 → 2022–2024


In [ ]:
X_q1 = ts[FEATS_Q1].values
y_q1 = ts['casos'].values

# ── Split A: Aleatorio ────────────────────────────────────────────────────────
X_tr_a, X_te_a, y_tr_a, y_te_a = train_test_split(
    X_q1, y_q1, test_size=0.2, random_state=SEED)

res_q1_a = comparar_regresion(X_tr_a, X_te_a, y_tr_a, y_te_a,
                               'Split Aleatorio', 'Q1 — Regresión Temporal')

# ── Split B: Temporal ─────────────────────────────────────────────────────────
mask_train_q1 = ts['Año del hecho'] <= 2021
mask_test_q1  = ts['Año del hecho'] >= 2022
X_tr_b = X_q1[mask_train_q1]; y_tr_b = y_q1[mask_train_q1]
X_te_b = X_q1[mask_test_q1];  y_te_b = y_q1[mask_test_q1]

res_q1_b = comparar_regresion(X_tr_b, X_te_b, y_tr_b, y_te_b,
                               'Split Temporal', 'Q1 — Regresión Temporal')

In [ ]:
# ── Comparativa Q1: split A vs B ──────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Q1 — Comparativa de modelos de Regresión', fontsize=13, fontweight='bold')

metricas = ['MAE','RMSE','R2']
colores_splits = {'Split Aleatorio':'#2C3E6B', 'Split Temporal':'#E74C3C'}

for idx, metrica in enumerate(metricas):
    ax = axes[idx]
    ancho = 0.35
    x = np.arange(len(res_q1_a))
    
    vals_a = res_q1_a[metrica].values
    vals_b = res_q1_b[metrica].reindex(res_q1_a.index, fill_value=0).values
    
    bars_a = ax.bar(x - ancho/2, vals_a, ancho, label='Split Aleatorio', color='#2C3E6B', alpha=0.85)
    bars_b = ax.bar(x + ancho/2, vals_b, ancho, label='Split Temporal',  color='#E74C3C', alpha=0.85)
    ax.set_title(metrica)
    ax.set_xticks(x)
    ax.set_xticklabels(res_q1_a.index, rotation=30, ha='right', fontsize=9)
    ax.spines[['top','right']].set_visible(False)
    if idx == 0:
        ax.legend()

plt.tight_layout()
plt.savefig('graficas/q1_comparativa_modelos.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Visualizar mejor modelo (RF) sobre la serie ───────────────────────────────
scaler_q1 = StandardScaler()
X_q1_sc = scaler_q1.fit_transform(X_q1)
rf_q1 = RandomForestRegressor(n_estimators=100, max_depth=8, random_state=SEED, n_jobs=-1)
rf_q1.fit(X_q1_sc[mask_train_q1], y_q1[mask_train_q1])
pred_rf_q1 = rf_q1.predict(X_q1_sc)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(ts['t'], ts['casos'], color='#2C3E6B', linewidth=1.8, label='Casos reales', zorder=3)
ax.plot(ts['t'][mask_test_q1], pred_rf_q1[mask_test_q1], color='#E74C3C',
        linewidth=2, linestyle='--', label='RF — Predicción 2022-2024', zorder=4)
ax.axvline(ts['t'][mask_test_q1].iloc[0], color='gray', linestyle=':', linewidth=1.5, label='Límite train/test')
ax.fill_between(ts['t'][mask_test_q1],
                pred_rf_q1[mask_test_q1]*0.9, pred_rf_q1[mask_test_q1]*1.1,
                alpha=0.15, color='#E74C3C')
ax.set_title('Q1 — Random Forest: predicción vs. casos reales (split temporal)', fontsize=12, fontweight='bold')
ax.set_xlabel('Mes (t)')
ax.set_ylabel('N° de casos')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{int(x):,}'))
ax.legend()
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('graficas/q1_rf_prediccion.png', dpi=150, bbox_inches='tight')
plt.show()

---
### Paso 5.2 — Q2: ¿La educación que se recibe disminuye el abuso intrafamiliar?

**Hipótesis:** A mayor nivel educativo de la víctima (`edu_n`), menor severidad del abuso registrado (`severidad_n`).  
**Target:** `severidad_n` (0=Leve, 1=Moderado, 2=Grave)  
**Features:** `edu_n` *(principal)* · `sexo_n` · `es_menor` · `ciclo_n` · `tiene_discapacidad` · `civil_n` *(controles)*  
**Modelos:** Reg. Logística · Árbol · Random Forest · SVM · K-NN · PyTorch MLP  
**Splits:** (A) Aleatorio 80/20 · (B) Temporal 2015–2021 → 2022–2024


In [ ]:
X_q2 = df_q2[FEATS_Q2].values
y_q2 = df_q2[TARGET].values

# Muestra para velocidad (SVM es O(n²) en tiempo)
MAX_SAMPLE = 80_000
if len(X_q2) > MAX_SAMPLE:
    idx_sample = np.random.choice(len(X_q2), MAX_SAMPLE, replace=False)
    X_q2_s = X_q2[idx_sample]; y_q2_s = y_q2[idx_sample]
    print(f"Muestra {MAX_SAMPLE:,} / {len(X_q2):,} registros para velocidad")
else:
    X_q2_s = X_q2; y_q2_s = y_q2

# ── Split A: Aleatorio estratificado ─────────────────────────────────────────
X_tr2_a, X_te2_a, y_tr2_a, y_te2_a = train_test_split(
    X_q2_s, y_q2_s, test_size=0.2, random_state=SEED, stratify=y_q2_s)

res_q2_a = comparar_clasificacion(X_tr2_a, X_te2_a, y_tr2_a, y_te2_a,
                                   'Split Aleatorio', 'Q2 — Perfil Víctima',
                                   fname_prefix='q2')

In [ ]:
# ── Split B: Temporal Q2 ─────────────────────────────────────────────────────
df_q2_train = df_q2[df_q2['Año del hecho'] <= 2021]
df_q2_test  = df_q2[df_q2['Año del hecho'] >= 2022]

if len(df_q2_train) > MAX_SAMPLE:
    df_q2_train = df_q2_train.sample(MAX_SAMPLE, random_state=SEED)
if len(df_q2_test) > MAX_SAMPLE//4:
    df_q2_test = df_q2_test.sample(MAX_SAMPLE//4, random_state=SEED)

X_tr2_b = df_q2_train[FEATS_Q2].values; y_tr2_b = df_q2_train[TARGET].values
X_te2_b = df_q2_test[FEATS_Q2].values;  y_te2_b = df_q2_test[TARGET].values
print(f"Train: {len(X_tr2_b):,} · Test: {len(X_te2_b):,}")

res_q2_b = comparar_clasificacion(X_tr2_b, X_te2_b, y_tr2_b, y_te2_b,
                                   'Split Temporal', 'Q2 — Perfil Víctima',
                                   fname_prefix='q2')

In [ ]:
# ── Comparativa Q2 ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Q2 — Comparativa de modelos de Clasificación (Perfil Víctima)', fontsize=13, fontweight='bold')

for idx, (res, split_name) in enumerate([(res_q2_a,'Split Aleatorio'),(res_q2_b,'Split Temporal')]):
    ax = axes[idx]
    f1_vals = res['F1 (pond.)'].sort_values(ascending=True)
    colores_bar = ['#E74C3C' if v == f1_vals.max() else '#2C3E6B' for v in f1_vals.values]
    bars = ax.barh(f1_vals.index, f1_vals.values, color=colores_bar, edgecolor='white')
    for bar, val in zip(bars, f1_vals.values):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=9)
    ax.set_title(split_name)
    ax.set_xlabel('F1 Score ponderado')
    ax.set_xlim(0, 1.05)
    ax.axvline(f1_vals.max(), color='#E74C3C', linestyle='--', linewidth=1, alpha=0.6)
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('graficas/q2_comparativa_modelos.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nMejor modelo Split Aleatorio:", res_q2_a['F1 (pond.)'].idxmax(),
      f"(F1={res_q2_a['F1 (pond.)'].max():.4f})")
print("Mejor modelo Split Temporal: ", res_q2_b['F1 (pond.)'].idxmax(),
      f"(F1={res_q2_b['F1 (pond.)'].max():.4f})")

---
### Paso 5.3 — Q3: ¿Los factores contextuales permiten clasificar la gravedad del hecho?

**Target:** `severidad_n` (0=Leve, 1=Moderado, 2=Grave) — mismo target que Q2 con features completamente distintas.  
La comparación Q2 vs Q3 responde: ¿protege más la **educación de la víctima** (Q2), o importa más el **contexto del hecho** (Q3)?  
**Features:** `hora_n`, `es_noche`, `zona_n`, `escenario_n`, `agresor_n`, `factor_n`, `mecanismo_n`  
**Modelos:** Reg. Logística · Árbol · Random Forest · SVM · K-NN · PyTorch MLP  
**Splits:** (A) Aleatorio 80/20 · (B) Temporal 2015–2021 → 2022–2024


In [ ]:
X_q3 = df_q3[FEATS_Q3].values
y_q3 = df_q3[TARGET].values

if len(X_q3) > MAX_SAMPLE:
    idx_s3 = np.random.choice(len(X_q3), MAX_SAMPLE, replace=False)
    X_q3_s = X_q3[idx_s3]; y_q3_s = y_q3[idx_s3]
else:
    X_q3_s = X_q3; y_q3_s = y_q3

# ── Split A ───────────────────────────────────────────────────────────────────
X_tr3_a, X_te3_a, y_tr3_a, y_te3_a = train_test_split(
    X_q3_s, y_q3_s, test_size=0.2, random_state=SEED, stratify=y_q3_s)

res_q3_a = comparar_clasificacion(X_tr3_a, X_te3_a, y_tr3_a, y_te3_a,
                                   'Split Aleatorio', 'Q3 — Factores Contextuales',
                                   fname_prefix='q3')

In [ ]:
# ── Split B: Temporal Q3 ─────────────────────────────────────────────────────
df_q3_train = df_q3[df_q3['Año del hecho'] <= 2021]
df_q3_test  = df_q3[df_q3['Año del hecho'] >= 2022]

if len(df_q3_train) > MAX_SAMPLE:
    df_q3_train = df_q3_train.sample(MAX_SAMPLE, random_state=SEED)
if len(df_q3_test) > MAX_SAMPLE//4:
    df_q3_test = df_q3_test.sample(MAX_SAMPLE//4, random_state=SEED)

X_tr3_b = df_q3_train[FEATS_Q3].values; y_tr3_b = df_q3_train[TARGET].values
X_te3_b = df_q3_test[FEATS_Q3].values;  y_te3_b = df_q3_test[TARGET].values

res_q3_b = comparar_clasificacion(X_tr3_b, X_te3_b, y_tr3_b, y_te3_b,
                                   'Split Temporal', 'Q3 — Factores Contextuales',
                                   fname_prefix='q3')

In [ ]:
# ── Comparativa Q3 ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Q3 — Comparativa de modelos (Factores Contextuales)', fontsize=13, fontweight='bold')

for idx, (res, split_name) in enumerate([(res_q3_a,'Split Aleatorio'),(res_q3_b,'Split Temporal')]):
    ax = axes[idx]
    f1_vals = res['F1 (pond.)'].sort_values(ascending=True)
    colores_bar = ['#E74C3C' if v == f1_vals.max() else '#2C3E6B' for v in f1_vals.values]
    bars = ax.barh(f1_vals.index, f1_vals.values, color=colores_bar, edgecolor='white')
    for bar, val in zip(bars, f1_vals.values):
        ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=9)
    ax.set_title(split_name)
    ax.set_xlabel('F1 Score ponderado')
    ax.set_xlim(0, 1.05)
    ax.axvline(f1_vals.max(), color='#E74C3C', linestyle='--', linewidth=1, alpha=0.6)
    ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('graficas/q3_comparativa_modelos.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nMejor modelo Split Aleatorio:", res_q3_a['F1 (pond.)'].idxmax())
print("Mejor modelo Split Temporal: ", res_q3_b['F1 (pond.)'].idxmax())

---
## Paso 6: Optimizar parámetros

### RandomizedSearchCV sobre el mejor modelo por pregunta

Una vez identificado el algoritmo ganador en el Paso 5, se optimizan sus hiperparámetros con `RandomizedSearchCV` (20 iteraciones, validación cruzada estratificada de 3 folds). Se elige búsqueda aleatoria sobre grid exhaustivo por la escala del dataset (~80–236 k registros).

El modelo ganador en Q2 y Q3 fue **Random Forest**; en Q1 **Random Forest Regressor**. Se optimizan:
- `n_estimators` (número de árboles)
- `max_depth` (profundidad máxima)
- `min_samples_leaf` (mínimo de muestras por hoja)
- `max_features` (features consideradas por split)


In [ ]:
# ── Q2: optimizar Random Forest ──────────────────────────────────────────────
print("Optimizando Random Forest para Q2...")
param_dist_rf = {
    'n_estimators':  [100, 200, 300],
    'max_depth':     [6, 8, 10, 12, None],
    'min_samples_leaf': [1, 3, 5, 10],
    'max_features':  ['sqrt', 'log2', 0.5],
    'class_weight':  ['balanced', 'balanced_subsample']
}
cv_strat = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

scaler_opt = StandardScaler()
X_opt = scaler_opt.fit_transform(X_q2_s)

rscv_q2 = RandomizedSearchCV(
    RandomForestClassifier(random_state=SEED, n_jobs=-1),
    param_dist_rf, n_iter=20, cv=cv_strat,
    scoring='f1_weighted', random_state=SEED, n_jobs=-1, verbose=0
)
rscv_q2.fit(X_opt, y_q2_s.astype(int))

print(f"\nMejores parámetros Q2:")
for k, v in rscv_q2.best_params_.items():
    print(f"  {k}: {v}")
print(f"F1 CV (best): {rscv_q2.best_score_:.4f}")

In [ ]:
# ── Q3: optimizar Random Forest ──────────────────────────────────────────────
print("Optimizando Random Forest para Q3...")
X_opt3 = scaler_opt.fit_transform(X_q3_s)

rscv_q3 = RandomizedSearchCV(
    RandomForestClassifier(random_state=SEED, n_jobs=-1),
    param_dist_rf, n_iter=20, cv=cv_strat,
    scoring='f1_weighted', random_state=SEED, n_jobs=-1, verbose=0
)
rscv_q3.fit(X_opt3, y_q3_s.astype(int))

print(f"\nMejores parámetros Q3:")
for k, v in rscv_q3.best_params_.items():
    print(f"  {k}: {v}")
print(f"F1 CV (best): {rscv_q3.best_score_:.4f}")

In [ ]:
# ── Q1: optimizar Random Forest Regressor ────────────────────────────────────
print("Optimizando Random Forest Regressor para Q1...")
param_dist_rfr = {
    'n_estimators': [100, 200, 300],
    'max_depth':    [4, 6, 8, 10, None],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.8]
}
scaler_q1_opt = StandardScaler()
X_q1_opt = scaler_q1_opt.fit_transform(X_q1)

rscv_q1 = RandomizedSearchCV(
    RandomForestRegressor(random_state=SEED, n_jobs=-1),
    param_dist_rfr, n_iter=15, cv=KFold(n_splits=5, shuffle=True, random_state=SEED),
    scoring='r2', random_state=SEED, n_jobs=-1, verbose=0
)
rscv_q1.fit(X_q1_opt, y_q1)

print(f"\nMejores parámetros Q1:")
for k, v in rscv_q1.best_params_.items():
    print(f"  {k}: {v}")
print(f"R² CV (best): {rscv_q1.best_score_:.4f}")

---
## Paso 7: Evaluar desempeño

### Métricas de evaluación

| Problema | Métricas usadas | Por qué |
|---|---|---|
| **Q1 — Regresión** | R², MAE, RMSE | R² mide cuánta varianza explica el modelo; MAE/RMSE cuantifican el error absoluto en número de casos |
| **Q2/Q3 — Clasificación** | F1 ponderado, AUC | F1 ponderado balancea precisión y recall en las 3 clases (leve/moderado/grave); AUC mide discriminación global |

Las **matrices de confusión** complementan las métricas agregadas: revelan si el modelo falla más en distinguir casos leves de moderados, o moderados de graves.

Para Q2 en particular, un F1 alto valida la hipótesis de que el nivel educativo **sí tiene poder predictivo** sobre la severidad del abuso. Si el F1 temporal ≈ F1 aleatorio, significa que la relación educación–severidad es **estable en el tiempo** y no es un artefacto del período analizado.

### Resumen comparativo


In [ ]:
# ── Tabla resumen final ───────────────────────────────────────────────────────
print("\n" + "="*70)
print("RESUMEN FINAL — MEJOR MODELO POR PREGUNTA Y SPLIT")
print("="*70)

resumen = {
    'Q1 Regresión (Split Aleatorio)': {
        'Mejor modelo': res_q1_a['R2'].idxmax(),
        'R²':  f"{res_q1_a['R2'].max():.4f}",
        'MAE': f"{res_q1_a['MAE'].min():.2f}",
        'RMSE':f"{res_q1_a['RMSE'].min():.2f}",
    },
    'Q1 Regresión (Split Temporal)': {
        'Mejor modelo': res_q1_b['R2'].idxmax(),
        'R²':  f"{res_q1_b['R2'].max():.4f}",
        'MAE': f"{res_q1_b['MAE'].min():.2f}",
        'RMSE':f"{res_q1_b['RMSE'].min():.2f}",
    },
    'Q2 Clasificación (Split Aleatorio)': {
        'Mejor modelo': res_q2_a['F1 (pond.)'].idxmax(),
        'F1 pond.': f"{res_q2_a['F1 (pond.)'].max():.4f}",
        'Accuracy': f"{res_q2_a['Accuracy'].max():.4f}",
    },
    'Q2 Clasificación (Split Temporal)': {
        'Mejor modelo': res_q2_b['F1 (pond.)'].idxmax(),
        'F1 pond.': f"{res_q2_b['F1 (pond.)'].max():.4f}",
        'Accuracy': f"{res_q2_b['Accuracy'].max():.4f}",
    },
    'Q3 Clasificación (Split Aleatorio)': {
        'Mejor modelo': res_q3_a['F1 (pond.)'].idxmax(),
        'F1 pond.': f"{res_q3_a['F1 (pond.)'].max():.4f}",
        'Accuracy': f"{res_q3_a['Accuracy'].max():.4f}",
    },
    'Q3 Clasificación (Split Temporal)': {
        'Mejor modelo': res_q3_b['F1 (pond.)'].idxmax(),
        'F1 pond.': f"{res_q3_b['F1 (pond.)'].max():.4f}",
        'Accuracy': f"{res_q3_b['Accuracy'].max():.4f}",
    },
}

df_resumen = pd.DataFrame(resumen).T.fillna('—')
print(df_resumen.to_string())
print("\n" + "="*70)

In [ ]:
# ── Gráfica resumen — F1 y R² por pregunta y split ───────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Resumen comparativo — Mejor F1 / R² por pregunta', fontsize=13, fontweight='bold')

# Q1
metricas_q1 = {'Split Aleatorio': res_q1_a['R2'].max(),
               'Split Temporal':  res_q1_b['R2'].max()}
axes[0].bar(metricas_q1.keys(), metricas_q1.values(),
            color=['#2C3E6B','#E74C3C'], edgecolor='white', width=0.5)
axes[0].set_title('Q1 — Regresión\n(métrica: R²)')
_max0 = max(metricas_q1.values())
axes[0].set_ylim(0, _max0 * 1.25)
for i, (k, v) in enumerate(metricas_q1.items()):
    axes[0].text(i, v + _max0 * 0.03, f'{v:.4f}', ha='center', fontweight='bold')
axes[0].spines[['top','right']].set_visible(False)

# Q2
metricas_q2 = {'Split Aleatorio': res_q2_a['F1 (pond.)'].max(),
               'Split Temporal':  res_q2_b['F1 (pond.)'].max()}
axes[1].bar(metricas_q2.keys(), metricas_q2.values(),
            color=['#2C3E6B','#E74C3C'], edgecolor='white', width=0.5)
axes[1].set_title('Q2 — Clasificación (perfil víctima)\n(métrica: F1 ponderado)')
_max1 = max(metricas_q2.values())
axes[1].set_ylim(0, _max1 * 1.25)
for i, (k, v) in enumerate(metricas_q2.items()):
    axes[1].text(i, v + _max1 * 0.03, f'{v:.4f}', ha='center', fontweight='bold')
axes[1].spines[['top','right']].set_visible(False)

# Q3
metricas_q3 = {'Split Aleatorio': res_q3_a['F1 (pond.)'].max(),
               'Split Temporal':  res_q3_b['F1 (pond.)'].max()}
axes[2].bar(metricas_q3.keys(), metricas_q3.values(),
            color=['#2C3E6B','#E74C3C'], edgecolor='white', width=0.5)
axes[2].set_title('Q3 — Clasificación (contexto)\n(métrica: F1 ponderado)')
_max2 = max(metricas_q3.values())
axes[2].set_ylim(0, _max2 * 1.25)
for i, (k, v) in enumerate(metricas_q3.items()):
    axes[2].text(i, v + _max2 * 0.03, f'{v:.4f}', ha='center', fontweight='bold')
axes[2].spines[['top','right']].set_visible(False)

plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig('graficas/resumen_final.png', dpi=150, bbox_inches='tight')
plt.show()
plt.close()

---
## Paso 8: Evaluar variables faltantes e importancia de features

### Objetivos
1. **Importancia de features** (Gini impurity via Random Forest): ¿qué tan dominante es `edu_n` en Q2 respecto a las variables de control? ¿y qué variables explican más la severidad en Q3?
2. **Variables del dataset no utilizadas**: ¿hay columnas relacionadas con educación o contexto socioeconómico que no se incluyeron pero podrían mejorar el modelo?

Este paso retroalimenta el Paso 3: si la importancia de `edu_n` es muy alta en Q2, confirma la hipótesis central. Si es baja, sugiere que la educación sola no es suficiente para explicar la severidad del abuso.


In [ ]:
# ── Importancia Q2 (perfil víctima) ──────────────────────────────────────────
scaler_imp = StandardScaler()
X_tr2_imp = scaler_imp.fit_transform(df_q2[FEATS_Q2].values)
rf_imp_q2 = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=SEED, n_jobs=-1)
rf_imp_q2.fit(X_tr2_imp, df_q2[TARGET].values.astype(int))
importancias_q2 = pd.Series(rf_imp_q2.feature_importances_, index=FEATS_Q2).sort_values(ascending=True)

# ── Importancia Q3 (contexto) ────────────────────────────────────────────────
X_tr3_imp = scaler_imp.fit_transform(df_q3[FEATS_Q3].values)
rf_imp_q3 = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=SEED, n_jobs=-1)
rf_imp_q3.fit(X_tr3_imp, df_q3[TARGET].values.astype(int))
importancias_q3 = pd.Series(rf_imp_q3.feature_importances_, index=FEATS_Q3).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Importancia de variables (Random Forest)', fontsize=13, fontweight='bold')

for ax, importancias, titulo, feats in [
    (axes[0], importancias_q2, 'Q2 — Perfil de la Víctima', FEATS_Q2),
    (axes[1], importancias_q3, 'Q3 — Factores Contextuales', FEATS_Q3)
]:
    colores = ['#E74C3C' if v == importancias.max() else '#2C3E6B' for v in importancias.values]
    ax.barh(importancias.index, importancias.values, color=colores, edgecolor='white')
    ax.set_title(titulo)
    ax.set_xlabel('Importancia (Gini)')
    ax.spines[['top','right']].set_visible(False)
    for i, (feat, val) in enumerate(importancias.items()):
        ax.text(val + 0.002, i, f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('graficas/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Variables del dataset no incluidas — análisis de potencial ───────────────
vars_no_usadas = [c for c in df.columns 
                  if c not in FEATS_Q2 + FEATS_Q3 + ['severidad_n','mes_num','t','mes_sin','mes_cos',
                                                       'trimestre','es_pandemia','es_post_pandemia',
                                                       'escenario_str','edu_n','sexo_n','es_menor',
                                                       'ciclo_n','tiene_discapacidad','civil_n','civil_n',
                                                       'hora_n','es_noche','zona_n','escenario_n',
                                                       'agresor_n','factor_n','mecanismo_n','edad_n',
                                                       'es_indigena']]

print("Variables del dataset no utilizadas como features:")
for col in sorted(vars_no_usadas):
    n_validos = (~df[col].astype(str).isin({'Sin información','No Sabe / No Informa','No aplica'})).sum()
    pct = n_validos / len(df) * 100
    estado = '✓ Usable' if pct > 60 else ('⚠ Parcial' if pct > 30 else '✗ Alta pérdida')
    print(f"  [{estado:12s}] {col:<50s} {pct:5.1f}% válido")

print("\n--- Variables con potencial no explorado ---")
print("  'Contexto del Hecho':  podría enriquecer Q3 (violencia física, psicológica, sexual...)")
print("  'Orientación Sexual':   71%+ sin info → no usable confiablemente")
print("  'Identidad de Género':  misma limitación")
print("  'País de Nacimiento':   mayoría Colombia → baja varianza, poco aporte predictivo")
print("  'Municipio del hecho':  alta cardinalidad (1000+ municipios) → requiere target encoding")

---
## Paso 9: Ajustar el modelo — Evaluación final y conclusiones

### Desempeño consolidado por pregunta y split

En esta sección se integran los resultados de todos los modelos y splits para determinar el modelo final recomendado por pregunta, considerando tanto el rendimiento en split aleatorio como la capacidad de generalización temporal (split B).


### Interpretación de resultados

| Pregunta | Reflexión |
|---|---|
| **Q1** | Si R² split temporal < R² aleatorio: el modelo no generaliza bien al futuro → las features temporales tienen límites para predecir más allá del patrón histórico |
| **Q2** | Si F1 es alto y `edu_n` domina la importancia de features: **la educación sí es un factor protector** — la hipótesis se confirma. Si F1 temporal ≈ F1 aleatorio, la relación es estable en el tiempo y no depende del período |
| **Q3** | Si Q3 F1 > Q2 F1: el *contexto del hecho* importa más que la *educación de la víctima* para determinar la gravedad. Si Q2 > Q3: la educación tiene mayor poder protector que el entorno |
| **PyTorch vs RF** | Si RF > MLP: en datos tabulares estructurados los árboles suelen ganar por su manejo natural de variables categóricas ordinales como `edu_n` |

### Limitaciones identificadas
- `Días de Incapacidad Medicolegal` tiene un porcentaje de valores sin información → sesgo en el target
- La educación registrada es la de la víctima al momento del hecho, no necesariamente el nivel alcanzado antes del abuso → posible causalidad inversa
- Variables socioeconómicas como ingreso o acceso a servicios no están en el dataset, lo que limita aislar el efecto puro de la educación
- El split temporal tiene pocos datos de test (36 meses) → mayor varianza en las estimaciones
